### `/etc/sudoers` — where the authorization actually lives

In [ ]:
dennie@HP:~/linux-tut$ ls -l /etc/sudoers
-r--r----- 1 root root 1800 Jan 29  2024 /etc/sudoers
dennie@HP:~/linux-tut$ cat /etc/sudoers
cat: /etc/sudoers: Permission denied

**READ THE OUTPUT:** mode `440` — readable only by `root` (owner) and the `root` group. Not even world-readable. You, `dennie`, get flat-out denied trying to `cat` it — and that's correct, working-as-designed behavior, not a bug. This file defines who can run what as whom, and letting regular users read it would leak the whole authorization model (and any custom rules) to anyone on the box.

Instead of editing it directly, Ubuntu grants access via **group membership** — a single line in `/etc/sudoers` (readable only by root, but its effect is publicly visible) says roughly:
```shell
%sudo   ALL=(ALL:ALL) ALL
```
Meaning: "anyone in the `sudo` group may run any command as any user." Confirm you're in it:

In [ ]:
dennie@HP:~/linux-tut$ getent group sudo
sudo:x:27:dennie
dennie@HP:~/linux-tut$ groups dennie
dennie : dennie adm cdrom sudo dip plugdev users

That's _why_ `sudo` works for your `dennie` account — not because you're special, but because you're listed in the `sudo` group, group ID `27`. This is the actual mechanism of **privilege delegation**: instead of handing out the root password, an admin just runs `usermod -aG sudo newuser` once, and that user can now escalate — using _their own_ password, with every action logged under _their own_ name.

**GOTCHA (would-be WSL trap, but not the case here):** some WSL Ubuntu installs configure the first user with **passwordless sudo** via a file in `/etc/sudoers.d/`, for convenience. I checked yours:

In [ ]:
dennie@HP:~/linux-tut$ sudo -n true
sudo: a password is required

`-n` tells sudo "don't prompt, just tell me if I'd be let through." Yours came back requiring a password — good, that's the secure default, not the passwordless shortcut some WSL setups take.

---

**TICKET SCENARIO:** _"I ran a script and now half my `/etc` config is broken, and I don't know who else touched this box."_ **DIAGNOSIS PATH:**

1. `sudo -l` — under _your own_ account, this lists exactly what you personally are authorized to run as root/other users (this one _will_ prompt for your password interactively — try it in your own terminal).
2. `journalctl _COMM=sudo` (or `grep sudo /var/log/auth.log` on non-systemd setups) — every `sudo` invocation is logged with the real username and the exact command run. This is the audit trail root-only workflows don't give you.
3. Compare that against who's in the `sudo` group (`getent group sudo`) to see who's even capable of having made the change.

In [ ]:
dennie@HP:~/linux-tut$ sudo -l
[sudo] password for dennie: 
Matching Defaults entries for dennie on HP:
    env_reset, mail_badpass,
    secure_path=/usr/local/sbin\:/usr/local/bin\:/usr/sbin\:/usr/bin\:/sbin\:/bin\:/snap/bin,
    use_pty

User dennie may run the following commands on HP:
    (ALL : ALL) ALL

Read it as: _user_ dennie, on _host_ HP, may run commands **as** `(any user : any group)`, and the command allowed is `ALL` (anything). That matches the `%sudo ALL=(ALL:ALL) ALL` line from `/etc/sudoers` we couldn't `cat` directly — `sudo -l` is exactly how you're _supposed_ to read your own sudoers rule without needing root access to the raw file.

In [ ]:
dennie@HP:~/linux-tut$ grep sudo /var/log/auth.log
2026-08-23T17:01:53.113854+00:00 HP sudo:   dennie : a password is required ; TTY=pts/4 ; PWD=/home/dennie/linux-tut ; USER=root ; COMMAND=/usr/bin/true

This is the audit trail in action:
`USER=root` = who you escalated _to_. The username at the front (`dennie`) = who actually did it. `COMMAND=/usr/bin/true` = exactly what ran. That triplet (real identity, target identity, exact command) is what you lose completely if everyone just does `su -` and works as literal `root` all day — the logs would just say `root` did everything, with no way to tell which human.

### TRY NOW

Run these yourself (the first will prompt for **your** password — that's expected, and proves you're not root):

In [ ]:
dennie@HP:~/linux-tut$ sudo whoami
root
dennie@HP:~/linux-tut$ whoami
dennie

`sudo whoami` → `root`, then `whoami` → `dennie` — exactly the pattern I wanted you to see: briefly elevated, immediately back to yourself. No lingering root shell.

**WHAT TO LOOK FOR:** `sudo whoami` should print `root` — for that one command, you briefly _became_ root. But run plain `whoami` right after, and you're back to `dennie` immediately.
> That "briefly root, then back to yourself" pattern is the entire point of `sudo` over `su -`: minimum privilege, minimum time, full accountability.

---

### `su -` vs `sudo -i` vs `sudo -s` vs plain `sudo` command
Four ways to get root, four different footprints:
|Command|Becomes root?|Whose password?|Environment|
|---|---|---|---|
|`sudo command`|only for that one command|**yours**|stays yours|
|`sudo -s`|full root shell|**yours**|keeps _your_ `$HOME`, aliases, shell config|
|`sudo -i`|full root shell|**yours**|switches to _root's_ `$HOME` (`/root`), like a real root login|
|`su -`|full root shell|**root's own**|switches to root's `$HOME`, root's login|

**Support default:** prefer plain `sudo command` for anything you can do in one shot (99% of ticket work). Reach for `sudo -i` only when you genuinely need a sustained root session (e.g., a multi-step recovery procedure) — and even then, `exit` the moment you're done.

GOTCHA — try this one yourself:

In [ ]:
dennie@HP:~/linux-tut$ su -
Password: 
su: Authentication failure

On stock Ubuntu (including most WSL images), **`root` has no usable password set** — the account is locked by default, precisely to _force_ admins through `sudo` instead of `su`. Try it and see what you get (likely `su: Authentication failure`). If you ever wanted `su -` to work, you'd first have to `sudo passwd root` to set one — which itself only proves the point: even _enabling_ the old-school path requires going through sudo first.